[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/terrell/terrell.ipynb)

# Terrell Rotation

A body moving at nearly the speed of light is contracted along its motion, yet no photograph shows the contraction. A photograph records the light that reaches the camera at one instant, and light from the far parts of a passing body set out earlier than light from its near parts, when the body was further back. The delay and the contraction together turn the picture instead of squashing it: a passing cube is seen turned, showing the face that trails behind it, and a sphere still shows a circular outline. Terrell and Penrose pointed this out in 1959, more than half a century after the contraction itself.

Events, and the rays of light between them, are modelled as antivectors, directions in spacetime; a ray is null, `ray.squared() == 0`, and the directions of the camera's sky are the rays arriving at it. In the body's own frame nothing moves: there the camera, at `frame << event`, takes an ordinary photograph of a body at rest, each ray running straight from a point of the body to the camera, and the boost `frame >> rays` carries the photograph to the camera's frame. A boost is linear, so rays in one hyperplane through the camera, where `plane & ray == 0`, stay in one, and those rays are a circle on the sky: the boost maps circles on the sky to circles. The outline of a sphere is such a circle in its own frame, so it is one in every frame, and the whole photograph is the one taken at rest from where the camera is in the body's frame: turned, not squashed.

In the vector notation of special relativity, a line of sight at angle $\theta$ from the body's motion in the camera's frame is at $\theta'$ in the body's, with $\cos\theta' = \frac{\cos\theta + \beta}{1 + \beta\cos\theta}$: seen at right angles to its motion, the body is photographed from $\cos\theta' = \beta$, turned by $\arcsin\beta$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import STA
from examples.animation import show_animation
from examples.relativity.terrell import render

np.set_printoptions(precision=1, suppress=True)

ga = STA
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Events, and the rays of light between them, are antivectors: directions in spacetime, intrinsic.
Antivector = ga.gatype.antivector()
Rotor = ga.gatype.rotor()

# One unit along the camera's time, and unit directions of space.
time = -mv.t.dual()                                          # [] Antivector
space = stack([mv.x, mv.y, mv.z]).dual()                     # [3] Antivector
# The direction the body travels, the camera's line of sight to its track, and up.
travel, line_of_sight, up = space

## 1. The camera in the body's frame

The camera sits at the origin of its own frame and photographs the body when the light from the body's centre reaches it. That light set out from an event on the camera's past light cone, back along the line of sight as far as the track the body moves along. In the body's frame that event is the body's centre, and the camera is somewhere beside it: `frame << event` reads an event in the body's frame. Seen at right angles to its motion, the body finds the camera behind the perpendicular, by the speed as a fraction of the distance, and that is where its photograph is taken from.

In [ ]:
# The distance from the camera to the track, and the speeds of the cases, as fractions of the speed of light.
distance = 4.5
speeds = np.array([0.0, 0.6, 0.9])

In [ ]:
def boost(speeds: np.ndarray) -> Rotor:
    """The body's frame at each speed, a turn in the plane of the motion and time."""
    return ((travel & time).dual() * (np.arctanh(speeds) / 2)).exp()            # [...] Rotor


def sighting(distance: float, angles: np.ndarray) -> Antivector:
    """The events where the body's centre sends the light the camera receives along a line of sight at
    each angle from the motion: the ray back along the line of sight, scaled to reach the track."""
    sight = (mv.xy * (-angles / 2)).exp() >> travel                             # [...] Antivector
    return (sight - time) * (distance / (mv.y & sight))                         # [...] Antivector


def rest_camera(frame: Rotor, centre: Antivector) -> Antivector:
    """Where the camera is in the body's frame when it receives the light the body's centre sent from
    the event `centre`: the camera's event relative to it, seen from the body, without its time."""
    camera = -(frame << centre)                                                  # [...] Antivector
    return camera - (mv.t & camera) * time                                      # [...] Antivector


frame = boost(speeds)                                                          # [cases] Rotor
camera = rest_camera(frame, sighting(distance, np.pi / 2))                     # [cases] Antivector
# How far behind the perpendicular the camera is, as a fraction of its distance.
leaning = (mv.x & camera) / camera.squared().square_root()                    # [cases] Scalar
print("turned by, in degrees:", np.degrees(np.arcsin(-leaning.to_array())))

## 2. Photographs of a passing pair

In the body's frame the photograph is an ordinary one of a body at rest. Each point sends its ray straight to the camera, and a face of the cube, or a point of the sphere, shows where the camera is on its outer side. The outline of the sphere is where the rays from the camera graze it, a circle on the sphere square to the line to the camera. The boost carries every ray to the camera's frame, `frame >> rays`, where it arrives from a different direction, and the pair is photographed each time its centre is seen along the same lines of sight at every speed.

In [ ]:
# The sphere's radius and the cube's half-width, and how far above and below the centre of the pair their
# centres are. The samples along each edge of the cube, along each line of the sphere's grid and around its
# outline; the longitudes of the grid's meridians and the latitudes of its parallels.
radius = 1.0
half_width = 1.0
height = 1.6
n_edge = 16
n_grid = 64
n_outline = 96
longitudes = np.arange(8) * np.pi / 4
latitudes = np.array([-np.pi / 3, -np.pi / 6, 0, np.pi / 6, np.pi / 3])
# The lines of sight to the centre of the pair as it passes, as angles from the motion, and the duration
# of a frame.
n_frames = 72
sightings = np.linspace(3 * np.pi / 4, np.pi / 4, n_frames)
duration_ms = 60

In [ ]:
# One edge of the cube's top face, turned by quarter turns about the vertical into the face's boundary.
edge = up + travel + line_of_sight * np.linspace(-1, 1, n_edge, endpoint=False)  # [edge] Antivector
square = (mv.xy * (-np.arange(4)[:, None] * np.pi / 4)).exp() >> edge         # [4, edge] Antivector
# The top face turned onto each face: itself, upside down, the face ahead, the face behind, and the faces
# towards and away from the camera.
turns = (mv.zx * (-np.array([0, 2, 1, -1, 0, 0]) * np.pi / 4)
         + mv.yz * (-np.array([0, 0, 0, 0, 1, -1]) * np.pi / 4)).exp()          # [6] Rotor
faces = -height * up + half_width * (turns[:, None] >> square.reshape(4 * n_edge))   # [6, 4 * edge] Antivector
normals = turns >> up                                                          # [6] Antivector
face_centres = -height * up + half_width * normals                             # [6] Antivector

# The sphere's grid: directions turned up out of the motion by a latitude, then about the vertical.
sphere = height * up                                                           # [] Antivector
meridians = (mv.xy * (-longitudes[:, None] / 2)).exp() >> ((mv.zx * (np.linspace(-np.pi / 2, np.pi / 2, n_grid) / 2)).exp() >> travel)
parallels = (mv.xy * (-np.linspace(0, 2 * np.pi, n_grid) / 2)).exp() >> ((mv.zx * (latitudes[:, None] / 2)).exp() >> travel)
grid = sphere + radius * concatenate([meridians, parallels])                   # [lines, grid] Antivector

Each lane is one speed, from the top: at rest, at six tenths and at nine tenths of the speed of light, on a stereographic map of the camera's sky, which draws circles on the sky as circles. The cube's face ahead is red, the face behind blue, and the face towards the camera violet. At rest the pair shows the face it comes with, then its front, then the face behind as it leaves. At nine tenths of the speed of light it shows the face behind nearly all the way, the sphere's grid turned with it, and the sphere's outline stays round. This cell takes about five seconds, most of it drawing the frames.

In [ ]:
def rays(camera: Antivector, points: Antivector) -> Antivector:
    """The rays from points of a body at rest to the camera at rest beside it, in their frame."""
    offsets = camera - points                                                   # [...] Antivector
    # A ray takes as long as it is long.
    return offsets + time * offsets.squared().square_root()                     # [...] Antivector


def facing(camera: Antivector, points: Antivector, normals: Antivector) -> Scalar:
    """How far the camera is out along the outward normal at each point: the point shows where this is positive."""
    return normals | (camera - points)                                          # [...] Scalar


def outline(camera: Antivector, centre: Antivector, radius: float, angles: np.ndarray) -> Antivector:
    """Where the rays from the camera graze a sphere, at each angle around the line to the camera."""
    offset = camera - centre                                                    # [...] Antivector
    distance = offset.squared().square_root()                                   # [...] Scalar
    # The grazing points ring the line from the sphere's centre to the camera. The ring is built around the
    # direction to a camera straight in front, and turned onto the camera's, which is never straight behind.
    turn = ((offset / distance) * -line_of_sight).square_root()                 # [...] Rotor
    nearest = (radius / distance) * -line_of_sight + (1 - (radius / distance).squared()).square_root() * up
    circle = (mv.zx * (angles / 2)).exp() >> nearest[..., None]                # [..., angles] Antivector
    return centre + radius * (turn[..., None] >> circle)                        # [..., angles] Antivector


def frames():
    """Per frame, in every case: the rays from the cube's faces and how far each turns towards the camera,
    the rays from the sphere's grid and how far each of its points does, and the rays grazing the sphere."""
    angles = np.linspace(0, 2 * np.pi, n_outline + 1)
    for angle in sightings:
        camera = rest_camera(frame, sighting(distance, angle))                 # [cases] Antivector
        grazing = outline(camera, sphere, radius, angles)                      # [cases, outline + 1] Antivector
        yield (frame[:, None, None] >> rays(camera[:, None, None], faces),
               facing(camera[:, None], face_centres, normals),
               frame[:, None, None] >> rays(camera[:, None, None], grid),
               facing(camera[:, None, None], grid, grid - sphere),
               frame[:, None] >> rays(camera[:, None], grazing))


show_animation(render.animate_photographs(frames()), "terrell_rotation", duration_ms)

In [ ]:
# checks
# Seen at right angles to its motion, the body is photographed turned by the arcsine of the speed.
np.testing.assert_allclose(leaning.to_array(), -speeds, atol=1e-12)
# The sphere's outline is a circle on the sky at every speed: its rays, scaled to one unit of the camera's
# time, lie in one hyperplane through the camera, so the join of any four of them vanishes.
*_, grazing = next(iter(frames()))                                             # [cases, outline + 1] Antivector
points = grazing / (mv.t & grazing)                                            # [cases, outline + 1] Antivector
np.testing.assert_allclose((points[:, 0] & points[:, 20] & points[:, 50] & points[:, 70]).to_array(), 0, atol=1e-12)